## simple_triton example

This notebook illustrates how to use the simple_triton package to perform inference on tiles from a whole-slide image using a Triton inference server.

Notes for running:
- See https://github.com/PathologyDataScience/simple_triton for details on launching the triton server container and mounting the model repository notebook
- This notebook requires installation of `mil`, `glimr`, and `histomics_stream`
- Run this notebook in a container with `--network=host` so that it can reach the Triton container
- Mount your model repository directory to the triton container
- Load the model (below)

In [1]:
# install large_image with tile sources
!pip install histomics_stream 'large_image[tiff]' \
  scikit_image --find-links https://girder.github.io/large_image_wheels

# install simple_triton
!pip install ../../simple_triton

# install ray tune dependencies and mil
!pip install pyarrow
!pip install tabulate
!pip install ray
!pip install ../../glimr
!pip install ../../mil

Defaulting to user installation because normal site-packages is not writeable
Looking in links: https://girder.github.io/large_image_wheels


Defaulting to user installation because normal site-packages is not writeable
Processing /home/lac5440/simple_triton
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for simple_triton: filename=simple_triton-0.1.dev441+g3f85fba.d20240314-py3-none-any.whl size=30883 sha256=1fedb0ce9f926fe8cb56dfa16eb6818949ef767f1a22315ba895ce7b81685a38
  Stored in directory: /tmp/pip-ephem-wheel-cache-2bttp1_b/wheels/98/c4/7b/2a3753547f3062a82c82afa48a94ffe0cd0b85863a054c4703
Successfully built simple_triton
  Attempting uninstall: simple_triton
    Found existing installation: simple_triton 0.1.dev441+g3f85fba.d20240314
    Uninstalling simple_triton-0.1.dev441+g3f85fba.d20240314:
      Successfully uninstalled simple_triton-0.1.dev441+g3f85fba.d20240314
Defaulting to user installation because normal site-packages is not writeable


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Processing /home/lac5440/glimr
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


  Created wheel for glimr: filename=glimr-0.1.dev122+g815ca45-py3-none-any.whl size=21729 sha256=233ae0546f105eb1b1318a7cef96e10a0b111c80ee7b9a431236cd4f198fa5af
  Stored in directory: /tmp/pip-ephem-wheel-cache-2zi0_m46/wheels/bd/b6/4b/aa7d6a78c3ea85f3c561af5fa2328ebb6ad3866fa493f5645a
Successfully built glimr
  Attempting uninstall: glimr
    Found existing installation: glimr 0.1.dev122+g815ca45
    Uninstalling glimr-0.1.dev122+g815ca45:
      Successfully uninstalled glimr-0.1.dev122+g815ca45


Defaulting to user installation because normal site-packages is not writeable
Processing /home/lac5440/mil
  Preparing metadata (setup.py) ... done
  Created wheel for mil: filename=mil-0.0.1-py3-none-any.whl size=54624 sha256=696d44025d1e7b21ca360058f26a53b0621cf84730ade6c20adbbd7ddc8b0e18
  Stored in directory: /tmp/pip-ephem-wheel-cache-d76if89c/wheels/3f/7f/ff/37340b9860dc6a822d46598d60b983dcfda706371ceafcd137
Successfully built mil
  Attempting uninstall: mil
    Found existing installation: mil 0.0.1
    Uninstalling mil-0.0.1:
      Successfully uninstalled mil-0.0.1


## Run client with no GPUs

If running Triton and the client on the same machine, we want to stop the client tensorflow from consuming GPU resources. By default, TensorFlow maps nearly all available GPU memory.

In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf

assert len(tf.config.list_physical_devices("GPU")) == 0

2024-03-14 04:46:50.154033: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-03-14 04:46:50.203608: I tensorflow/core/platform/cpu_feature_guard.cc:183] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Download sample data

Download the hosted whole-slide image and mask.

In [3]:
import pooch

# download whole slide image and corresponding mask
# wsi_path = pooch.retrieve(
#     fname="TCGA-AN-A0G0-01Z-00-DX1.svs",
#     url="https://drive.google.com/uc?export=download&id=19agE_0cWY582szhOVxp9h3kozRfB4CvV&confirm=t&uuid=6f2d51e7-9366-4e98-abc7-4f77427dd02c&at=ALgDtswlqJJw1KU7P3Z1tZNcE01I:1679111148632",
#     known_hash="d046f952759ff6987374786768fc588740eef1e54e4e295a684f3bd356c8528f",
#     path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
# )
wsi_path = "/home/lac5440/TCGA-AN-A0G0-01Z-00-DX1.svs"
# mask_path = pooch.retrieve(
#     fname="TCGA-AN-A0G0-01Z-00-DX1.mask.png",
#     url="https://drive.google.com/uc?export=download&id=17GOOHbL8Bo3933rdIui82akr7stbRfta",
#     known_hash="bb657ead9fd3b8284db6ecc1ca8a1efa57a0e9fd73d2ea63ce6053fbd3d65171",
#     path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
# )
mask_path = "/home/lac5440/TCGA-AN-A0G0-01Z-00-DX1.mask.png"

## Create an encoder model

`tf_encoder` creates encoder models from the available models in `tf.keras.applications`. Running this takes time as the model is downloaded. The encoder model is saved into the designated triton server model respository that is mounted within the triton server container.

In [4]:
import numpy as np
from pprint import pprint
from simple_triton.encoders import tf_encoder
from simple_triton.model import TritonModel

# model parameters
keras_name = "EfficientNetV2S"
model_name = f"{keras_name}.tensorflow"  # set model name

# create the model and capture output dimensionality
if not os.path.exists(os.path.join("~/models", model_name)):
    dimension_output = tf_encoder(
        "~/models", keras_name, model_name, input_shape=(tile, tile, 3), pooling="avg"
    )

## Load the model using `TritonModel`

After creating the model, we load the model into Triton using the `TritonModel` class. This class contains methods for loading, unloading, and checking the status of models. To load the model we create a simple configuration with batch size 64, and allow Triton to generate the remaining configuration fields. By default it will load a single copy of the model on each system GPU, and will often automatically set optimizations like pinned memory.

In [5]:
# triton parameters
url = "localhost:8001"  # url for grpc access to tirton server

# load tensorflow model - set maximum batch size
model = TritonModel(model_name, url)
model.load(config={"maxBatchSize": 128})
assert model.is_loaded()
pprint(model.get_config())

{'backend': 'tensorflow',
 'defaultModelFilename': 'model.savedmodel',
 'dynamicBatching': {'preferredBatchSize': [128]},
 'input': [{'dataType': 'TYPE_FP32',
            'dims': ['224', '224', '3'],
            'name': 'input_2'}],
 'instanceGroup': [{'count': 1,
                    'gpus': [0, 1, 2, 3, 4, 5, 6, 7],
                    'kind': 'KIND_GPU',
                    'name': 'EfficientNetV2S.tensorflow'}],
 'maxBatchSize': 128,
 'name': 'EfficientNetV2S.tensorflow',
 'optimization': {'inputPinnedMemory': {'enable': True},
                  'outputPinnedMemory': {'enable': True}},
 'output': [{'dataType': 'TYPE_FP32', 'dims': ['1280'], 'name': 'avg_pool'}],
 'platform': 'tensorflow_savedmodel',
 'versionPolicy': {'latest': {'numVersions': 1}}}


## Advanced configuration with `ConfigBuilder`

The `ConfigBuilder` class provides access to advanced configuration options like backend optimizations. Here, we create a duplicate model on each GPU (`count=2`) and convert the model to mixed precision to improve speed and memory usage. The model is reloaded using this advanced configuration.

In [6]:
from simple_triton.config import ConfigBuilder

# initialize builder with a basic configuration
builder = ConfigBuilder(model_name, config={"maxBatchSize": 128})

# increase the number of model instances per GPU to 2
builder.add_instance_group(count=2)

# add automatic mixed precision
builder.add_mixed_precision()

# re-load model with new config
model.load(config=builder.config)

# print config
pprint(model.get_config())

{'backend': 'tensorflow',
 'defaultModelFilename': 'model.savedmodel',
 'dynamicBatching': {'preferredBatchSize': [128]},
 'input': [{'dataType': 'TYPE_FP32',
            'dims': ['224', '224', '3'],
            'name': 'input_2'}],
 'instanceGroup': [{'count': 2,
                    'gpus': [0, 1, 2, 3, 4, 5, 6, 7],
                    'kind': 'KIND_GPU',
                    'name': 'EfficientNetV2S.tensorflow_0'}],
 'maxBatchSize': 128,
 'name': 'EfficientNetV2S.tensorflow',
 'optimization': {'executionAccelerators': {'gpuExecutionAccelerator': [{'name': 'auto_mixed_precision'}]},
                  'inputPinnedMemory': {'enable': True},
                  'outputPinnedMemory': {'enable': True}},
 'output': [{'dataType': 'TYPE_FP32', 'dims': ['1280'], 'name': 'avg_pool'}],
 'platform': 'tensorflow_savedmodel',
 'versionPolicy': {'latest': {'numVersions': 1}}}


## Run the inference

First, a histomics stream study is created defining the tiles that need to be read based on the whole-slide image, tissue mask, and desired magnification, tile size, and tile overlap. The chunk parameter is used to group tiles during disk reads to maximize throughput. This study initializes a `LargeimagePrefetch` iterator that generates batches of tiles and tile metadata using prefetching.

This iterator is passed to the inference function that is parameterized by the number of tiles per batch, the number of workers, and the maximum number of pending inferences per worker.

In [8]:
from simple_triton.feature_extraction import inference, study
from simple_triton.tile_iterators import LargeimagePrefetch
from simple_triton.utils import analyze
from time import time

# slide parameters
batch = 64
magnification = 20.
tile = 224
chunk = 896
mask_threshold = 0.5

# create a histomics-stream study from a wsi/mask pair
hs_study = study(
    (wsi_path, mask_path), t=(tile, tile), chunk=(chunk, chunk), 
    objective=magnification, mask_threshold=mask_threshold
)

# tile iterator parameters
batch = 64
prefetch = 4
workers = 16  # total number of tile 
icc = True # apply ICC color correction

# inference parameters
limit = 10  # limit on number of pending requests per worker
verbose = True  # display inference statistics and debugging information

# start timer
start = time()

# create tile iterator
iterator = LargeimagePrefetch(hs_study, icc, batch, prefetch, workers)



# # analyze performance
# analyze(times)

In [9]:
# inference
#features, tile_info, times, failures = inference(
batches = inference(
    iterator,
    model_name,
    url="localhost:8001",
    limit=limit
)

# display elapsed time
print(f"Total elapsed time: {time()-start}")

Total elapsed time: 40.588250160217285


In [11]:
batches[0]

{'model_name': 'EfficientNetV2S.tensorflow',
 'metadata': [{'overlap_height': 0,
   'overlap_width': 0,
   'filename': b'/home/lac5440/TCGA-AN-A0G0-01Z-00-DX1.svs',
   'slide_name': b'TCGA-AN-A0G0-01Z-00-DX1.svs',
   'slide_group': b'TCGA-AN-A0G0-01Z-00-DX1.svs',
   'chunk_width': 896,
   'chunk_height': 896,
   'target_magnification': 20.0,
   'scan_magnification': 40.0,
   'read_magnification': 40.0,
   'returned_magnification': 20.0,
   'level': 7,
   'slide_width': 27607,
   'slide_height': 20572,
   'slide_height_tiles': 91,
   'slide_width_tiles': 123,
   'mask_height': 642,
   'mask_width': 862,
   'chunk_top': 672,
   'chunk_left': 21280,
   'chunk_bottom': 1568,
   'chunk_right': 22176,
   'tile_top': 672,
   'tile_left': 21280},
  {'overlap_height': 0,
   'overlap_width': 0,
   'filename': b'/home/lac5440/TCGA-AN-A0G0-01Z-00-DX1.svs',
   'slide_name': b'TCGA-AN-A0G0-01Z-00-DX1.svs',
   'slide_group': b'TCGA-AN-A0G0-01Z-00-DX1.svs',
   'chunk_width': 896,
   'chunk_height': 89

## Write features to .tfr

In [ ]:
from mil.io.reader import read_record, peek
from mil.io.writer import write_record
import tensorflow as tf

# concatenate features
features = np.concatenate(features[0], axis=0)

# create dummy labels
labels = {"labels": np.random.uniform(size=(10))}

# write to tfrecord
write_record(
    "./triton.tfr", features, tile_info, labels, structured=False, precision=tf.float16
)

# get list of .tfr variables for de-serialization
serialized = list(tf.data.TFRecordDataset(["./triton.tfr"]))[0]
variables = peek(serialized)

# verify reading
read_record(serialized, variables, structured=False, precision=tf.float16)